In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import torch
from scipy import ndimage as ndi
from skimage.segmentation import watershed
from skimage.feature import peak_local_max
from torchmetrics.detection import IntersectionOverUnion
from torchmetrics import Dice
from skimage.filters import threshold_multiotsu
from torchmetrics.classification import Precision, Recall, F1Score, JaccardIndex
from sklearn.metrics import (
    precision_score, recall_score, f1_score, jaccard_score,
    confusion_matrix, classification_report
)


# main multi-Otsu 

In [ ]:
def otsu_multi_thresholding(image, num_classes=3):
    thresholds = threshold_multiotsu(image, classes=num_classes)
    segmented_image = np.digitize(image, bins=thresholds)
    return segmented_image

def segment_input_image(image_path, num_classes=3):
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    segmented = otsu_multi_thresholding(image, num_classes=num_classes)
    return segmented

def compute_iou_torchmetrics_detection(predictions, ground_truth, num_classes, device="cpu"):
    iou_metric = IntersectionOverUnion(class_metrics=False).to(device)
    preds_list = []
    targets_list = []
    for c in range(num_classes):
        pred_mask = torch.tensor(predictions == c, dtype=torch.uint8, device=device).unsqueeze(0)
        gt_mask = torch.tensor(ground_truth == c, dtype=torch.uint8, device=device).unsqueeze(0)
        preds_list.append({"masks": pred_mask})
        targets_list.append({"masks": gt_mask})
    iou_value = iou_metric(preds_list, targets_list)
    return float(iou_value["iou"].mean().cpu()) * 100

def evaluate_segmentation(predictions, ground_truth, num_classes=3, device="cpu"):
    predictions = predictions.flatten()
    ground_truth = ground_truth.flatten()
    labels = np.arange(num_classes)

    IoU_percent = compute_iou_torchmetrics_detection(
        predictions.reshape(-1, 1), ground_truth.reshape(-1, 1), num_classes, device=device
    )

    jaccard_macro = jaccard_score(
        ground_truth, predictions, labels=labels, average="macro", zero_division=0
    ) * 100
    precision = precision_score(
        ground_truth, predictions, labels=labels, average="macro", zero_division=0
    ) * 100
    recall = recall_score(
        ground_truth, predictions, labels=labels, average="macro", zero_division=0
    ) * 100
    f1 = f1_score(
        ground_truth, predictions, labels=labels, average="macro", zero_division=0
    ) * 100

    # Dice metric using torchmetrics
    dice_metric = Dice(num_classes=num_classes, average="macro").to(device)
    dice_value = dice_metric(
        torch.tensor(predictions, dtype=torch.int64, device=device),
        torch.tensor(ground_truth, dtype=torch.int64, device=device)
    ).item() * 100

    cm = confusion_matrix(ground_truth, predictions, labels=labels)
    report = classification_report(
        ground_truth, predictions, labels=labels,
        target_names=[f"Class {i}" for i in labels], zero_division=0
    )

    metrics = {
        "IoU (%)": IoU_percent,
        "Jaccard Score (%)": jaccard_macro,
        "Precision (%)": precision,
        "Recall (%)": recall,
        "F1-score (%)": f1,
        "Dice Metric (%)": dice_value,
        "Confusion Matrix": cm,
        "Classification Report": report
    }
    return metrics

def process_images(input_dir, ground_truth_dir, num_classes=3, device="cpu", save_csv=True):
    input_files = sorted(os.listdir(input_dir))
    gt_files = sorted(os.listdir(ground_truth_dir))
    results = []
    print(f"\nProcessing {len(input_files)} image pairs...\n")

    for img_file, gt_file in zip(input_files, gt_files):
        input_path = os.path.join(input_dir, img_file)
        gt_path = os.path.join(ground_truth_dir, gt_file)
        image = cv2.imread(input_path, cv2.IMREAD_GRAYSCALE)
        ground_truth = cv2.imread(gt_path, cv2.IMREAD_GRAYSCALE)
        segmented = otsu_multi_thresholding(image, num_classes=num_classes)
        metrics = evaluate_segmentation(segmented, ground_truth, num_classes=num_classes, device=device)
        metrics["Image"] = img_file
        results.append(metrics)
        print(f"✔ Processed {img_file}")

    df = pd.DataFrame([
        {
            "Image": r["Image"],
            "IoU (%)": r["IoU (%)"],
            "Jaccard Score (%)": r["Jaccard Score (%)"],
            "Precision (%)": r["Precision (%)"],
            "Recall (%)": r["Recall (%)"],
            "F1-score (%)": r["F1-score (%)"],
            "Dice Metric (%)": r["Dice Metric (%)"]
        }
        for r in results
    ])

    mean_row = df.mean(numeric_only=True)
    mean_row["Image"] = "MEAN"
    df = pd.concat([df, pd.DataFrame([mean_row])], ignore_index=True)

    if save_csv:
        csv_path = os.path.join(input_dir, "multi_otsu_metrics.csv")
        df.to_csv(csv_path, index=False)
        print(f"\n✅ Results saved to: {csv_path}\n")

    print("\n===== Mean Metrics =====")
    print(df.tail(1).to_string(index=False))
    return df

if __name__ == "__main__":
    input_dir = "path/to/your/input_images"
    ground_truth_dir = "path/to/your/ground_truths"
    num_classes = 3
    device = "cuda" if torch.cuda.is_available() else "cpu"
    results_df = process_images(input_dir, ground_truth_dir, num_classes=num_classes, device=device,_


# Watershed 

In [ ]:
def watershed_segmentation(image, grad_thresh=0.18, min_basin_size=50):
    image = cv2.GaussianBlur(image, (3, 3), 0)
    gradient = cv2.Laplacian(image, cv2.CV_64F)
    gradient = cv2.convertScaleAbs(gradient)
    grad_norm = gradient / 255.0
    mask = grad_norm > grad_thresh
    distance = ndi.distance_transform_edt(~mask)
    coords = peak_local_max(distance, footprint=np.ones((3, 3)), labels=~mask)
    mask_markers = np.zeros(distance.shape, dtype=bool)
    mask_markers[tuple(coords.T)] = True
    markers, _ = ndi.label(mask_markers)
    labels = watershed(-distance, markers, mask=~mask)
    unique, counts = np.unique(labels, return_counts=True)
    for val, cnt in zip(unique, counts):
        if cnt < min_basin_size:
            labels[labels == val] = 0
    ret, thresh = cv2.threshold(image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    segmented = np.zeros_like(image, dtype=np.uint8)
    segmented[labels == 0] = 0
    segmented[(labels > 0) & (image > ret)] = 1
    segmented[(labels > 0) & (image <= ret)] = 2
    return segmented

def compute_metrics(y_true, y_pred, num_classes, device="cpu"):
    metrics = {}
    y_true = torch.tensor(y_true, dtype=torch.int64, device=device)
    y_pred = torch.tensor(y_pred, dtype=torch.int64, device=device)
    iou_metric = IntersectionOverUnion(class_metrics=False).to(device)
    preds_list = []
    targets_list = []
    for c in range(num_classes):
        pred_mask = (y_pred == c).to(torch.uint8).unsqueeze(0)
        gt_mask = (y_true == c).to(torch.uint8).unsqueeze(0)
        preds_list.append({"masks": pred_mask})
        targets_list.append({"masks": gt_mask})
    iou_value = iou_metric(preds_list, targets_list)
    metrics["IoU (%)"] = float(iou_value["iou"].mean().cpu()) * 100
    jaccard_macro = JaccardIndex(task="multiclass", num_classes=num_classes, average="macro").to(device)
    metrics["Jaccard Score (%)"] = jaccard_macro(y_pred, y_true).item() * 100
    precision = Precision(task="multiclass", num_classes=num_classes, average="macro").to(device)
    recall = Recall(task="multiclass", num_classes=num_classes, average="macro").to(device)
    f1 = F1Score(task="multiclass", num_classes=num_classes, average="macro").to(device)
    metrics["Precision (%)"] = precision(y_pred, y_true).item() * 100
    metrics["Recall (%)"] = recall(y_pred, y_true).item() * 100
    metrics["F1-score (%)"] = f1(y_pred, y_true).item() * 100
    dice_metric = Dice(num_classes=num_classes, average="macro").to(device)
    metrics["Dice Metric (%)"] = dice_metric(y_pred, y_true).item() * 100
    return metrics

def process_images_watershed(input_dir, ground_truth_dir, grad_thresh=0.18, min_basin_size=50, num_classes=3, device="cpu", max_images=None):
    input_files = sorted(os.listdir(input_dir))
    gt_files = sorted(os.listdir(ground_truth_dir))
    if max_images:
        input_files = input_files[:max_images]
        gt_files = gt_files[:max_images]
    results = []
    cm_total = np.zeros((num_classes, num_classes), dtype=np.int64)
    for in_file, gt_file in zip(input_files, gt_files):
        img_path = os.path.join(input_dir, in_file)
        gt_path = os.path.join(ground_truth_dir, gt_file)
        image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        gt = cv2.imread(gt_path, cv2.IMREAD_GRAYSCALE)
        if image is None or gt is None:
            print(f"Skipping {in_file}")
            continue
        segmented = watershed_segmentation(image, grad_thresh=grad_thresh, min_basin_size=min_basin_size)
        metrics = compute_metrics(gt, segmented, num_classes, device)
        results.append(metrics)
        cm_total += confusion_matrix(gt.flatten(), segmented.flatten(), labels=np.arange(num_classes))
        print(f"\nProcessed {in_file}")
        for k, v in metrics.items():
            print(f"{k}: {v:.2f}")
    mean_metrics = {k: np.mean([r[k] for r in results]) for k in results[0]}
    print("\n===== Mean Metrics =====")
    for k, v in mean_metrics.items():
        print(f"{k}: {v:.2f}")
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm_total, annot=True, fmt='d', cmap='Blues', xticklabels=[0,1,2], yticklabels=[0,1,2])
    plt.xlabel('Predicted')
    plt.ylabel('Ground Truth')
    plt.title('Confusion Matrix')
    plt.tight_layout()
    plt.show()
    return mean_metrics, cm_total

if __name__ == "__main__":
    input_dir = r"D:\University\Master\Thesis\naturally fractured coal reservoir ((Dataset))\Naturally Fractured coal Rocks\cropped images"
    ground_truth_dir = r"D:\University\Master\Thesis\naturally fractured coal reservoir ((Dataset))\Naturally Fractured coal Rocks\more_accurate_labeled_zero"
    device = "cuda" if torch.cuda.is_available() else "cpu"
    mean_metrics, cm = process_images_watershed(
        input_dir,
        ground_truth_dir,
        grad_thresh=0.18,
        min_basin_size=50,
        num_classes=3,
        device=device,
        max_images=2092
    )
